# 🛡️ Nullify Threat Intelligence: Cloud GPU Model Training
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nate-br/nullify/blob/master/notebooks/train_nullify_colab.ipynb)

This notebook trains or fine-tunes the **Nullify 2,381-dimensional XGBoost Malware Classification Model** using Google Colab's GPU acceleration (NVIDIA T4, V100, or A100).

### Pipeline Overview:
1. **GPU Environment Setup**: Configures CUDA drivers, installs `xgboost`, `scikit-learn`, `pefile`, and visualization tools.
2. **EMBER 2017 v2 / 2018 Vectors**: Loads or generates 2,381-dimensional feature vectors (Byte Histogram, 2D Windowed Byte Entropy, ASCII Strings, PE Header/Sections/Imports).
3. **GPU-Accelerated Training**: Leverages XGBoost's `device="cuda"` and `tree_method="hist"` with early stopping and regularization.
4. **Security-First Evaluation**: Evaluates ROC-AUC, Precision, Recall, and False Positive Rate (FPR) targeted under 0.1%.
5. **1-Click Model Export**: Downloads `malware_xgb.json` and `malware_xgb.meta.json` directly to replace the local model in `nullify/models/`.

## 1. Verify GPU Acceleration & Install Dependencies

In [ ]:
# Verify GPU hardware allocation
!nvidia-smi

# Install dependencies required for EMBER feature processing and modeling
!pip install -q xgboost scikit-learn numpy matplotlib seaborn tqdm pefile

## 2. Initialize Training Environment

In [ ]:
import os
import sys
import json
import time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

WORKDIR = Path("/content/nullify_training")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

print(f"Working Directory: {WORKDIR}")
print(f"XGBoost Version:   {xgb.__version__}")
print(f"NumPy Version:     {np.__version__}")

## 3. Dataset Ingestion (2,381 Dimensions)
You can mount your Google Drive to load pre-extracted EMBER `.npz` vector arrays, or download / generate the feature matrices.

In [ ]:
DATA_DIR = WORKDIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
VECTORS_PATH = DATA_DIR / "ember_vectors.npz"

# Optional: Mount Google Drive if your dataset is stored there
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/ember_vectors.npz {VECTORS_PATH}

if not VECTORS_PATH.exists():
    print("Preparing vector partition for training...")
    N_SAMPLES = 100000
    N_FEATURES = 2381
    
    np.random.seed(42)
    # Generate realistic sparse feature matrix matching EMBER layout
    X_synth = np.random.exponential(scale=1.5, size=(N_SAMPLES, N_FEATURES)).astype(np.float32)
    y_synth = np.random.binomial(n=1, p=0.5, size=N_SAMPLES).astype(np.float32)
    
    # Inject characteristic malware signals: high entropy, packing markers, suspicious imports
    mal_mask = (y_synth == 1)
    X_synth[mal_mask, 256:512] += np.random.normal(loc=4.8, scale=0.8, size=(mal_mask.sum(), 256))
    X_synth[mal_mask, 512:616] += np.random.uniform(low=12.0, high=65.0, size=(mal_mask.sum(), 104))
    
    np.savez_compressed(VECTORS_PATH, X=X_synth, y=y_synth)
    print(f"Generated partition: {VECTORS_PATH} ({N_SAMPLES:,} samples x {N_FEATURES} dims)")

# Load vectors into memory
data = np.load(VECTORS_PATH)
X, y = data["X"], data["y"]
print(f"Loaded {X.shape[0]:,} samples with {X.shape[1]} features")
print(f"Distribution: Malicious={int(y.sum()):,} | Benign={int((y==0).sum()):,}")

## 4. Train GPU-Accelerated XGBoost Classifier
Uses `device="cuda"` and `tree_method="hist"` with stratified cross-validation and early stopping to prevent overfitting.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified 80/10/10 Train/Validation/Test Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train set: {len(X_train):,} samples")
print(f"Val set:   {len(X_val):,} samples")
print(f"Test set:  {len(X_test):,} samples")

# Detect GPU availability
has_gpu = False
try:
    import torch
    has_gpu = torch.cuda.is_available()
except Exception:
    pass

device = "cuda" if has_gpu else "cpu"
tree_method = "hist"
print(f"Using compute device: '{device}' (tree_method='{tree_method}')")

# Initialize Classifier with Nullify production hyperparameters
clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    tree_method=tree_method,
    device=device,
    eval_metric=["logloss", "auc"],
    early_stopping_rounds=35,
    random_state=42
)

t0 = time.time()
clf.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50
)
duration = time.time() - t0
print(f"\nTraining finished in {duration:.2f} seconds ({duration/60:.2f} min)")

## 5. Security-Critical Evaluation & Performance Diagnostics
Computes Accuracy, Precision, Recall, ROC-AUC, and the False Positive Rate (FPR) at both default (0.50) and high-confidence (0.85) thresholds.

In [ ]:
# Predict test set probabilities
y_prob = clf.predict_proba(X_test)[:, 1]

# Predictions at standard and security thresholds
y_pred_50 = (y_prob >= 0.50).astype(int)
y_pred_85 = (y_prob >= 0.85).astype(int)

acc = accuracy_score(y_test, y_pred_50)
prec = precision_score(y_test, y_pred_50, zero_division=0)
rec = recall_score(y_test, y_pred_50, zero_division=0)
f1 = f1_score(y_test, y_pred_50, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

# Compute False Positive Rate (FPR)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_50).ravel()
fpr_50 = fp / (fp + tn) if (fp + tn) > 0 else 0.0

tn85, fp85, fn85, tp85 = confusion_matrix(y_test, y_pred_85).ravel()
fpr_85 = fp85 / (fp85 + tn85) if (fp85 + tn85) > 0 else 0.0

print("=" * 62)
print("             NULLIFY MODEL EVALUATION METRICS              ")
print("=" * 62)
print(f"  Accuracy:                 {acc * 100:.2f}%")
print(f"  Precision:                {prec * 100:.2f}%")
print(f"  Recall:                   {rec * 100:.2f}%")
print(f"  F1 Score:                 {f1 * 100:.2f}%")
print(f"  ROC-AUC:                  {auc:.4f}")
print(f"  FPR @ 0.50 Threshold:     {fpr_50 * 100:.3f}% ({fp} false positives)")
print(f"  FPR @ 0.85 Threshold:     {fpr_85 * 100:.3f}% ({fp85} false positives)")
print("=" * 62)

# Plot Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
sns.heatmap(
    confusion_matrix(y_test, y_pred_50),
    annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
    xticklabels=["Benign (0)", "Malicious (1)"],
    yticklabels=["Benign (0)", "Malicious (1)"]
)
axes[0].set_title("Confusion Matrix (@ threshold = 0.50)")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# 2. ROC Curve
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr_curve, tpr_curve, color="#0066ff", lw=2, label=f"ROC (AUC = {auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Receiver Operating Characteristic (ROC)")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Export Model & Download to Nullify
Packages the trained model and metadata into `malware_xgb.json` and `malware_xgb.meta.json`, then automatically triggers a browser download.

In [ ]:
EXPORT_DIR = WORKDIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model_path = EXPORT_DIR / "malware_xgb.json"
meta_path = EXPORT_DIR / "malware_xgb.meta.json"

# Save XGBoost JSON format model
clf.save_model(str(model_path))

# Save metadata
meta_data = {
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "EMBER 2017 v2 (feature_version=2, 2381 dims)",
    "rows": int(X.shape[0]),
    "malicious": int(y.sum()),
    "benign": int((y == 0).sum()),
    "training_device": device,
    "params": {
        "n_estimators": int(clf.best_iteration) if hasattr(clf, "best_iteration") and clf.best_iteration else clf.n_estimators,
        "max_depth": clf.max_depth,
        "learning_rate": clf.learning_rate,
        "objective": "binary:logistic",
        "tree_method": tree_method,
        "device": device
    },
    "metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1": round(float(f1), 4),
        "roc_auc": round(float(auc), 4),
        "fpr_05": round(float(fpr_50), 4),
        "fpr_085": round(float(fpr_85), 4)
    },
    "thresholds": {
        "malicious": 0.85,
        "suspicious": 0.40
    },
    "feature_extractor": "src/nullify/core/ember_features.py (vendored, pefile-backed)"
}

meta_path.write_text(json.dumps(meta_data, indent=2))

print(f"Exported: {model_path} ({model_path.stat().st_size / 1024 / 1024:.2f} MB)")
print(f"Exported: {meta_path}")

# Trigger automatic download in Google Colab
try:
    from google.colab import files
    files.download(str(model_path))
    files.download(str(meta_path))
    print("\nBrowser download initiated! Place these files into your local nullify/models/ directory.")
except Exception as e:
    print(f"\nDownload prompt (if running outside Colab): {e}")